# BankSim Fraud Detection Analysis

This notebook explains and orchestrates the reusable `fraud_pipeline` package. It does not own cleaning, splitting, historical-feature, training, evaluation, or export business logic.

## 1. Problem Definition

The objective is to identify fraudulent transactions under severe class imbalance. False negatives leave fraud undetected; false positives create review cost and customer friction. Precision, recall, F1, F2, PR-AUC, and ROC-AUC are therefore primary measures, while accuracy is supporting context.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from fraud_pipeline.cleaning import clean_transactions
from fraud_pipeline.config import SPLIT_CONFIG
from fraud_pipeline.data import load_raw_transactions, source_metadata
from fraud_pipeline.features import engineer_features, model_dataset
from fraud_pipeline.splitting import chronological_split, summarize_splits
from fraud_pipeline.training import build_preprocessor, load_frozen_model_bundle


## 2. Dataset

BankSim `step` is the zero-based simulated **day** index. The source spans days 0–179, approximately six simulated months. There is no intraday timestamp, so hour-of-day cannot be recovered or derived.

In [ ]:
raw_df = load_raw_transactions()
source_metadata(raw_df), raw_df.head(), raw_df['fraud'].value_counts(normalize=True).sort_index()


## 3. Data Cleaning

Cleaning strips source quoting, enforces the canonical schema and types, and validates missing values, target values, amounts, and simulated-day values. The implementation lives in `fraud_pipeline.cleaning`.

In [ ]:
cleaned_df = clean_transactions(raw_df)
cleaned_df.info()


## 4. Exploratory Analysis

Exploration is descriptive only; it does not define pipeline transformations.

In [ ]:
daily_summary = cleaned_df.groupby('step').agg(
    transactions=('fraud', 'size'),
    fraud_transactions=('fraud', 'sum'),
    fraud_amount=('amount', lambda values: values[cleaned_df.loc[values.index, 'fraud'].eq(1)].sum()),
)
daily_summary['fraud_rate'] = daily_summary['fraud_transactions'] / daily_summary['transactions']
daily_summary.head()


## 5. Chronological Split

The single split definition lives in `fraud_pipeline.config.SPLIT_CONFIG`. Chronological separation prevents later simulated days from influencing earlier training decisions.

In [ ]:
clean_splits = chronological_split(cleaned_df)
summarize_splits(clean_splits)
SPLIT_CONFIG.rule


## 6. Historical / Behavioral Feature Engineering

For a transaction on simulated day T, all historical features use only days `< T`. Transactions on day T never see one another because BankSim provides no reliable within-day ordering. Entity and relationship definitions are shared with the training-history artifact through `fraud_pipeline.history`.

In [ ]:
engineered_df = engineer_features(cleaned_df)
model_df = model_dataset(engineered_df)
engineered_df.shape, model_df.shape


## 7. Preprocessing

A future corrected retrain uses standardized numeric features and one-hot categorical features. Raw `step` remains available for splitting but is removed from model predictors. No `day = step // 24` or `hour_of_day = step % 24` features are created.

In [ ]:
model_splits = chronological_split(model_df)
X_train = model_splits.train.drop(columns=['fraud', 'step'])
y_train = model_splits.train['fraud']
corrected_preprocessor = build_preprocessor(X_train)
len(X_train), len(X_train.columns)


## 8. Feature Selection

The current production Random Forest uses 57 RF-selected processed features. The legacy temporal fields are not among them. Any new feature selection belongs to a separate, reviewed retraining experiment.

In [ ]:
frozen = load_frozen_model_bundle()
len(frozen.selected_feature_names), frozen.selected_feature_names[:10]


## 9. Random Forest Training

This structural refactor deliberately loads the frozen Random Forest. It does not fit or overwrite a model. The explicit training APIs in `fraud_pipeline.training` are reserved for a separately approved methodology run.

In [ ]:
frozen.model.get_params(), frozen.model.n_features_in_


## 10. Threshold Selection

The frozen operational threshold was selected on validation data and is not re-optimized here.

In [ ]:
frozen.threshold


## 11. Validation Performance

Validation metrics are loaded from the immutable evaluation export rather than recomputed during an ordinary compatibility build.

In [ ]:
import json
from utils.paths import RESULTS_DIR
saved_metrics = json.loads((RESULTS_DIR / 'final_metrics.json').read_text())
saved_metrics['validation']


## 12. Final Test Performance

The held-out final test remains separate from training and threshold selection.

In [ ]:
saved_metrics['test']


## 13. Decision Evidence

Model Decision Evidence reports Random Forest fraud probability, signed threshold margin, tree agreement, tree-probability dispersion, and reached-leaf training support. Historical Context separately reports factual entity and behavior familiarity from the training partition. Historical Context is **not another ML model** and does not estimate correctness.

> **RETIRED EXPERIMENT — NOT USED BY APPLICATION:** the previous Logistic Regression correctness model and P95-normalized support outputs remain only under legacy research artifacts.

In [ ]:
from utils.data_loader import load_validation_transaction
from utils.inference import assess_transaction
sample = load_validation_transaction(396332)
assess_transaction(sample)


## 14. Dashboard Artifact Export

One compatibility build owns processed data, split, dashboard, validation-demo, history, and manifest exports. It loads the frozen model and cannot retrain it.

In [ ]:
from fraud_pipeline.build import build_compatibility_artifacts
REBUILD_ACTIVE_ARTIFACTS = False
if REBUILD_ACTIVE_ARTIFACTS:
    build_compatibility_artifacts()


## 15. Final Conclusions

The active data path is reproducible, chronological, day-semantic, and leakage-controlled. The current frozen model is preserved through an explicit compatibility adapter because its old preprocessor declares two invalid temporal inputs; neither reaches the fitted Random Forest. Corrected-model retraining remains a separate methodology decision.